# Customer Churn Prediction — Part 1: Exploratory Data Analysis (EDA)
**Project Title:** Customer Churn Prediction Using Machine Learning with Model Optimization and Deployment-Ready Pipeline

### Objectives of this Notebook:
1. Load the IBM Telco Customer Churn dataset.
2. Perform data hygiene (detecting nulls, data type casting, white-space values in `TotalCharges`).
3. Analyze target class balance and demographic/account distributions.
4. Visualize relationship between features and churn (Contract, Tenure, Charges, Tech Support, Internet Service).
5. Derive strategic business insights.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

# 1. Load Dataset
df = pd.read_csv('../data/dataset.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### 2. Data Quality & Type Checking
We inspect missing values, data types, and check for hidden whitespace in `TotalCharges`.

In [2]:
print("Column Info & Data Types:")
print(df.info())

# Check whitespace in TotalCharges
empty_spaces = (df['TotalCharges'].astype(str).str.strip() == '').sum()
print(f"\nEmpty string rows in TotalCharges: {empty_spaces}")

# TotalCharges should be float. Customers with tenure=0 haven't had their first monthly bill.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce').fillna(0.0)
print(f"Total missing values after coercion: {df.isnull().sum().sum()}")

### 3. Target Variable Analysis: Churn Distribution
Analyzing class skewness in the target variable.

In [3]:
churn_counts = df['Churn'].value_counts()
churn_rate = (churn_counts['Yes'] / len(df)) * 100

print(f"No Churn: {churn_counts['No']} ({100 - churn_rate:.2f}%)")
print(f"Churn:    {churn_counts['Yes']} ({churn_rate:.2f}%)")

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Churn', palette=['#3b82f6', '#ef4444'])
plt.title('Target Variable Distribution: Customer Churn')
plt.xlabel('Churn Status')
plt.ylabel('Customer Count')
plt.show()

### 4. Categorical Driver: Contract Type vs Churn
Contract duration is one of the strongest predictive determinants in subscription services.

In [4]:
plt.figure(figsize=(8, 4.5))
contract_churn = df.groupby(['Contract', 'Churn']).size().unstack(fill_value=0)
contract_churn_pct = contract_churn.div(contract_churn.sum(axis=1), axis=0) * 100
contract_churn_pct.plot(kind='bar', stacked=True, color=['#3b82f6', '#ef4444'], figsize=(8, 4.5))
plt.title('Churn Rate by Contract Duration (%)')
plt.ylabel('Percentage')
plt.xlabel('Contract Type')
plt.xticks(rotation=0)
plt.show()

### 5. Continuous Drivers: Tenure and Monthly Charges
Observing customer lifecycle hazard rates and pricing friction.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tenure distribution
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', palette=['#3b82f6', '#ef4444'], ax=axes[0])
axes[0].set_title('Tenure Distribution by Churn Status')
axes[0].set_xlabel('Tenure (Months)')

# Monthly Charges KDE
sns.kdeplot(data=df[df['Churn']=='No']['MonthlyCharges'], label='Retained', color='#3b82f6', fill=True, ax=axes[1])
sns.kdeplot(data=df[df['Churn']=='Yes']['MonthlyCharges'], label='Churned', color='#ef4444', fill=True, ax=axes[1])
axes[1].set_title('Monthly Charges Density by Churn Status')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

### Key EDA Conclusions:
1. **Contract Type:** Month-to-month contracts have >42% churn risk, whereas 2-year contracts drop to <3% churn.
2. **Tenure Dynamics:** Churn is heavily concentrated in the first 0-12 months of tenure (onboarding hazard).
3. **Charges:** Churners display significantly higher monthly charges ($70 - $100 range) compared to retained customers ($20 - $60 range).
4. **Internet Technology:** Fiber optic customers churn at higher rates than DSL users when Tech Support/Online Security add-ons are absent.